## 🎓 연습문제

### 데이터 준비

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

fish = pd.read_csv('./data/fish.csv')

fish_input = fish[['Weight','Length','Diagonal','Height','Width']]
fish_target = fish['Species']

train_input, test_input, train_target, test_target = train_test_split(fish_input, fish_target, random_state=42)

ss = StandardScaler()
ss.fit(train_input)    
train_scaled = ss.transform(train_input)  
test_scaled = ss.transform(test_input)

### 문제 1

loss='hinge' 로 바꿔 학습해보고 정확도 비교하시오.

In [8]:
from sklearn.linear_model import SGDClassifier

sc = SGDClassifier(loss='log_loss', max_iter=100, random_state=42, tol=None)
sc.fit(train_scaled, train_target)
print(f"실제epoch수 : {sc.n_iter_}")
print(f"loss_훈련정확도 : {sc.score(train_scaled, train_target)}")
print(f"loss_테스트정확도 : {sc.score(test_scaled, test_target)}")

sc = SGDClassifier(loss='hinge', max_iter=100, random_state=42, tol=None)
sc.fit(train_scaled, train_target)
print(f"hinge_훈련정확도 : {sc.score(train_scaled, train_target)}")
print(f"hinge_테스트정확도 : {sc.score(test_scaled, test_target)}")

실제epoch수 : 100
loss_훈련정확도 : 0.957983193277311
loss_테스트정확도 : 0.925
hinge_훈련정확도 : 0.9495798319327731
hinge_테스트정확도 : 0.925


### 문제 2

epoch를 50, 100, 300으로 바꿔 비교하시오.

In [25]:
import numpy as np
classes = np.unique(train_target)

def train_with_epochs(n_epochs):
    sc = SGDClassifier(loss='log_loss', random_state=42, shuffle=False)
    for _ in range(n_epochs):
        sc.partial_fit(train_scaled, train_target, classes=classes)
    train_acc = sc.score(train_scaled, train_target)
    test_acc  = sc.score(test_scaled, test_target)
    print(f"epochs={n_epochs:<3} | train acc={train_acc:.4f} | test acc={test_acc:.4f}")

for e in [30, 50, 100, 300]:
    train_with_epochs(e)

epochs=30  | train acc=0.8319 | test acc=0.8500
epochs=50  | train acc=0.8824 | test acc=0.9000
epochs=100 | train acc=0.9328 | test acc=0.9250
epochs=300 | train acc=0.9580 | test acc=0.9250


In [26]:
sc = SGDClassifier(loss='log_loss', max_iter=50, random_state=42, tol=None, shuffle=False)
sc.fit(train_scaled, train_target)
print(f"실제epoch수 : {sc.n_iter_}")
print(f"loss_훈련정확도 : {sc.score(train_scaled, train_target)}")
print(f"loss_테스트정확도 : {sc.score(test_scaled, test_target)}")

실제epoch수 : 50
loss_훈련정확도 : 0.8739495798319328
loss_테스트정확도 : 0.9


In [ ]:
# partial_fit 방식
# 데이터 전체를 한 번 학습
# → epoch 1

# 다시 전체 데이터 학습
# → epoch 2
# ...
# SGD 업데이트가
# epoch마다 이어서 계속 진행

# 가중치가 계속 이어짐
# 데이터 셔플 없음
# tol 같은 조기 종료 없음

###############
# fit(max_iter=50)
# 1. 매 epoch마다 데이터 shuffle
# 2. mini update 수행
# 3. tol 체크 (여기서는 tol=None이라 없음)
# 4. 반복
# epoch마다 데이터 순서가 바뀜

### 문제 3

learning_rate='constant'로 바꾸고 eta0를 조정해보시오.

In [29]:
def train_with_eta0(eta0, epochs=200):
    classes = np.unique(train_target)
    sc = SGDClassifier(
        loss='log_loss',
        learning_rate='constant',
        eta0=eta0,
        random_state=42
    )
    for _ in range(epochs):
        sc.partial_fit(train_scaled, train_target, classes=classes)
    train_acc = sc.score(train_scaled, train_target)
    test_acc  = sc.score(test_scaled, test_target)
    print(f"eta0={eta0:<6} | train acc={train_acc:.4f} | test acc={test_acc:.4f}")

for eta in [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]:
    train_with_eta0(eta, epochs=200)

eta0=0.0001 | train acc=0.6218 | test acc=0.7000
eta0=0.001  | train acc=0.7311 | test acc=0.8000
eta0=0.01   | train acc=0.7983 | test acc=0.8500
eta0=0.1    | train acc=0.8908 | test acc=0.9250
eta0=1      | train acc=0.9244 | test acc=0.9000
eta0=10     | train acc=0.7311 | test acc=0.7000
eta0=100    | train acc=0.5630 | test acc=0.5250


In [ ]:
# 초기 학습률 = 0.01
# 가중치를 업데이트할 때 0.01 × 기울기 만큼 이동
# eta0 : 초기 학습률 => 경사하강법에서 한 번에 이동하는 크기
# learning_rate :constant(eta0 그대로 사용)

# - `eta0`가 **너무 크면** 손실이 줄지 않고 **발산**하거나 정확도가 불안정해질 수 있다.
# - `eta0`가 **너무 작으면** 학습이 매우 느려서 **과소적합**이 될 수 있다.
# - 적절한 `eta0`에서 test 정확도가 가장 높게 나온다.